# 流特征提取

In [9]:
import os
from scapy.utils import rdpcap
from scapy.layers.inet import IP, TCP
from scapy.layers.tls import *
import torch  # For tensor storage

## 前置处理
根据 summary.txt 获取有效的五元组: `(src_ip, src_port, dst_ip, dst_port)`
为了兼容双向流，可以保证 `src_ip < dst_ip`

In [5]:
# 过滤pcap文件，只提取指定四元组的报文
def build_vaild_flow_ids(summary_file):
    vaild_flow_ids = []
    with open(summary_file, "r") as f:
        lines = f.readlines()
        # 转换为四元组
        for line in lines:
            flow = eval(line)
            src_ip, src_port, dst_ip, dst_port = flow
            flow_id = (src_ip, src_port, dst_ip, dst_port)
            reverse_flow_id = (dst_ip, dst_port, src_ip, src_port)
            vaild_flow_ids.append(min(flow_id, reverse_flow_id))
    return vaild_flow_ids

# print(vaild_ids)



## 流特征提取 & 计算

流的各维度特征计算，目前单流纬度特征包括：
1. 包长序列
2. 包负载长度序列
3. 包标记位序列
4. 上行/下行
5. 包时间戳序列
6. 握手包内容 
7. 待完善

In [ ]:
# Helper function to extract flow identifier
def get_flow_id(packet):
    ip_layer = packet[IP]
    tcp_layer = packet[TCP]
    return (ip_layer.src, tcp_layer.sport, ip_layer.dst, tcp_layer.dport)


# 获取双向流的ID
def get_bidirectional_flow_id(packet):
    ip_layer = packet[IP]
    tcp_layer = packet[TCP]
    # Create a flow identifier
    flow_id = (ip_layer.src, tcp_layer.sport, ip_layer.dst, tcp_layer.dport)
    reverse_flow_id = (ip_layer.dst, tcp_layer.dport, ip_layer.src, tcp_layer.sport)
    # Return the lexicographically smaller tuple to ensure consistency
    return min(flow_id, reverse_flow_id)


# Helper function to extract packet length
def get_packet_length(packet):
    return len(packet)


# Helper function to extract packet timestamp
def get_packet_timestamp(packet):
    return packet.time


# Helper function to extract TCP flags
def get_tcp_flags(packet):
    return packet[TCP].flags


# Helper function to extract Payload Len
def get_payload_length(packet):
    if TCP in packet:
        # Get the payload of the TCP layer
        payload = packet[TCP].payload
        # Return the length of the payload
        return len(payload)
    return 0  # Return 0 if no payload exists


# Helper function to determine if a packet is uplink or downlink
def is_uplink(packet, flow_id):
    """
    Determine if a packet is uplink (client to server) or downlink (server to client).

    Args:
        packet: A Scapy packet object.
        flow_id: A tuple (src_ip, src_port, dst_ip, dst_port) representing the flow.

    Returns:
        str: "uplink" if the packet is client to server, "downlink" if server to client.
    """
    src_ip, src_port, dst_ip, dst_port = flow_id

    # Check if the packet matches the uplink direction
    if packet[IP].src == src_ip and packet[TCP].sport == src_port:
        return "uplink"  # Client to server

    # Check if the packet matches the downlink direction
    if packet[IP].src == dst_ip and packet[TCP].sport == dst_port:
        return "downlink"  # Server to client

    return "unknown"  # If it doesn't match either direction


# 提取流的 握手包内容, 需要具体内容
def extract_handshake_payload(packet):
    # Read packets from the pcap file
    handshake_packets = []
    # Check if the packet has IP and TCP layers
    if IP in packet and TCP in packet:
        tcp_layer = packet[TCP]
        # Check for SYN and ACK flags
        if tcp_layer.flags & 0x02:
            handshake_packets.append(packet)
        elif tcp_layer.flags & 0x10:
            handshake_packets.append(packet)
    # Return the handshake packets
    return handshake_packets


# 提取流信息的函数
def extract_flows(pcap_file, extract_features=None, vaild_flow_ids=None):
    """
    Extract packet length sequences for each TCP flow from a pcap file.

    Args:
        pcap_file (str): Path to the pcap file.
        extract_features (list): List of features to extract from packets.

    Returns:
        dict: A dictionary where keys are flow identifiers (e.g., tuple of IPs and ports)
              and values are lists of packet lengths.
    """
    if not os.path.exists(pcap_file):
        raise FileNotFoundError(f"PCAP file not found: {pcap_file}")

    # Read packets from the pcap file
    packets = rdpcap(pcap_file)

    flows = {}  # Dictionary to store flows and their packet lengths
    # 切分成流, 流纬度特征提取
    for packet in packets:
        # Check if the packet has IP and TCP layers
        if IP in packet and TCP in packet:
            flow_id = get_bidirectional_flow_id(packet)
            if vaild_flow_ids != None and flow_id not in vaild_flow_ids:
                continue
            raw_flow_id = tuple(flow_id)
            flow_id = str(flow_id)
            if flow_id not in flows:
                flows[flow_id] = {}
            if "length" in extract_features:
                flows[flow_id].setdefault("length", []).append(get_packet_length(packet))
            if "timestamp" in extract_features:
                flows[flow_id].setdefault("timestamp", []).append(get_packet_timestamp(packet))
            if "flags" in extract_features:
                flows[flow_id].setdefault("flags", []).append(get_tcp_flags(packet))
            if "handshake" in extract_features:
                handshake_payload = extract_handshake_payload(packet)
                flows[flow_id]["handshake"].extend(handshake_payload)
            if "payload_length" in extract_features:
                flows[flow_id].setdefault("payload_length", []).append(get_payload_length(packet))
            if "direction" in extract_features:
                flows[flow_id]["direction"] = is_uplink(packet, raw_flow_id)

    return flows

## 特征存储

特征的存储，目前支持的存储格式为：

1. Json格式
2. Tensor格式

In [7]:
import json


# 保存成 tensor 张量
def save_flows_as_tensors(flows, output_dir):
    """
    Save flows as tensors to the specified directory.

    Args:
        flows (dict): A dictionary where keys are flow identifiers and values are lists of packet info dicts.
        output_dir (str): Directory to save the tensors.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # for i, (flow_id, packets) in enumerate(flows.items()):
    # Convert packet info to a PyTorch tensor
    # tensor_data = {
    #     'length': [packet.get('length', 0) for packet in packets],
    #     'timestamp': [packet.get('timestamp', 0) for packet in packets],
    #     'flags': [packet.get('flags', 0) for packet in packets],
    #     'handshake_payload': [len(packet.get('handshake_payload', [])) for packet in packets],
    #     'payload_length': [packet.get('payload_length', 0) for packet in packets],
    #     'direction': [1 if packet.get('direction') == 'uplink' else 0 for packet in packets]
    # }
    # tensor = {key: torch.tensor(value, dtype=torch.float32) for key, value in tensor_data.items()}

    # Save the tensor to a file
    # flow_file = os.path.join(output_dir, f"flow_{i}.pt")
    # torch.save(tensor, flow_file)
    # print(f"Saved flow {flow_id} to {flow_file}")


def save_flows_feature_as_json(flows, output_dir):
    """
    Save flows as JSON files to the specified directory.

    Args:
        flows (dict): A dictionary where keys are flow identifiers and values are lists of packet info dicts.
        output_dir (str): Directory to save the JSON files.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    json.dump(
        flows,
        open(os.path.join(output_dir, "feature.json"), "w"),
        indent=4,
        separators=(",", ":"),
    )

## 全链路链路测试

执行前置处理、特征提取、特征计算、全链路测试

In [8]:
if __name__ == "__main__":

    data_dir = "/home/tanyongfeng.cy/Projects/Tantic/data/tans.fun/4f3ba313-c5cf-40ac-96d1-3b09570cf534"
    pcap_file = ""
    summary_file = ""
    # find pcap file and summary file
    for filename in os.listdir(data_dir):
        if filename == "traffic.pcap":
            pcap_file = os.path.join(data_dir, filename)
        elif filename == "summary.txt":
            summary_file = os.path.join(data_dir, filename)
    if pcap_file == "" or summary_file == "":
        raise FileNotFoundError("No pcap file or summary file found in the directory")

    vaild_flow_ids = build_vaild_flow_ids(summary_file)
    print(vaild_flow_ids)

    # Example usage
    output_dir = f"{data_dir}/feature"  # Directory to save tensors

    # Extract packet lengths for each TCP flow
    flows = extract_flows(
        pcap_file,
        extract_features=["length", "payload_length", "direction"],
        vaild_flow_ids=vaild_flow_ids,  # Optional: filter by valid flow IDs
    )

    # Save flows as tensors
    save_flows_feature_as_json(flows, output_dir)

NameError: name 'os' is not defined

In [ ]:
# 加载张量
tensor = torch.load("output_tensors/flow_1.pt")
print(tensor)  # 输出张量内容

{'length': tensor([55., 66., 55., 66.]), 'timestamp': tensor([0., 0., 0., 0.]), 'flags': tensor([0., 0., 0., 0.]), 'handshake_payload': tensor([0., 0., 0., 0.]), 'payload_length': tensor([1., 0., 1., 0.]), 'direction': tensor([0., 0., 0., 0.])}
